In [1]:
import sys
import platform
import torch
import transformers

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

print("\nMPS available:", torch.backends.mps.is_available())
print("MPS built:", torch.backends.mps.is_built())

if torch.backends.mps.is_available():
    print("MPS device:", torch.device("mps"))

print("\nCPU threads:", torch.get_num_threads())

try:
    import peft
    print("PEFT:", peft.__version__)
except ImportError:
    print("PEFT: Not installed")

try:
    import psutil
    memory = psutil.virtual_memory()
    print("Total system RAM (GB):", round(memory.total / 1024**3, 2))
    print("Available RAM (GB):", round(memory.available / 1024**3, 2))
except ImportError:
    print("psutil: Not installed")

Python: 3.11.8
Platform: macOS-26.6.2-arm64-arm-64bit
PyTorch: 2.15.0
Transformers: 4.57.6

MPS available: True
MPS built: True
MPS device: mps

CPU threads: 4
PEFT: Not installed
Total system RAM (GB): 8.0
Available RAM (GB): 2.46


In [4]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "peft"
])

print("PEFT installation complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 4.5 MB/s  0:00:00m ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [peft]━━━━━━ 1/2 [peft]
PEFT installation complete.


In [1]:
import torch
import transformers
import peft

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("MPS available:", torch.backends.mps.is_available())

PyTorch: 2.15.0
Transformers: 4.57.6
PEFT: 0.21.3
MPS available: True


In [3]:
import gc
import torch

from transformers import AutoModel, AutoTokenizer
from peft import LoraConfig, get_peft_model

MODEL_NAME = "xlm-roberta-base"
DEVICE = torch.device("cpu")

gc.collect()

if torch.backends.mps.is_available():
    torch.mps.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

encoder = AutoModel.from_pretrained(
    MODEL_NAME,
    attn_implementation="eager"
)

lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["query", "value"],
    lora_dropout=0.05,
    bias="none"
)

encoder = get_peft_model(
    encoder,
    lora_config
)

encoder.print_trainable_parameters()

print("LoRA encoder initialized.")
print("Current device:", DEVICE)

trainable params: 147,456 || all params: 278,191,104 || trainable%: 0.0530
LoRA encoder initialized.
Current device: cpu


In [5]:
total = sum(
    p.numel()
    for p in encoder.parameters()
)

trainable = sum(
    p.numel()
    for p in encoder.parameters()
    if p.requires_grad
)

print("Total parameters:", f"{total:,}")
print("Trainable parameters:", f"{trainable:,}")
print("Trainable percentage:", f"{100 * trainable / total:.4f}%")

assert trainable > 0
assert trainable < total * 0.05

print("\nLoRA parameter verification passed.")

Total parameters: 278,191,104
Trainable parameters: 147,456
Trainable percentage: 0.0530%

LoRA parameter verification passed.


In [7]:
import torch.nn as nn
import torch.nn.functional as F

class LoRAWSDClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        hidden = encoder.config.hidden_size

        self.dropout = nn.Dropout(0.15)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(256, num_classes)
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        target_mask,
        class_mask=None,
        labels=None
    ):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        hidden = outputs.last_hidden_state
        weights = target_mask.unsqueeze(-1)

        pooled = (
            (hidden * weights).sum(dim=1)
            / weights.sum(dim=1).clamp(min=1)
        )

        logits = self.classifier(
            self.dropout(pooled)
        )

        if class_mask is not None:
            logits = logits.masked_fill(
                ~class_mask.bool(), -1e4
            )

        if labels is None:
            return logits

        loss = F.cross_entropy(logits, labels)
        return loss, logits


SENSES = [
    "bank.1", "bank.2",
    "interest.1", "interest.2",
    "plant.1", "plant.2",
    "point.1", "point.2",
    "right.1", "right.2",
    "table.1", "table.2"
]

sense_to_id = {
    sense: i for i, sense in enumerate(SENSES)
}

model = LoRAWSDClassifier(
    encoder=encoder,
    num_classes=len(SENSES)
)

print("LoRA WSD classifier initialized.")
print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)

LoRA WSD classifier initialized.
Trainable parameters: 347404


In [9]:
import json
from pathlib import Path
from torch.utils.data import Dataset

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

V2 = ROOT / "artifacts" / "wsd_v2"

def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f if line.strip()
        ]

train_rows = read_jsonl(V2 / "train.jsonl")
val_rows = read_jsonl(V2 / "validation.jsonl")

assert len(train_rows) == 506
assert len(val_rows) == 60

class LoRAWSDDataset(Dataset):
    def __init__(self, rows, tokenizer, max_length=64):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        text = row["text_en"]
        start = int(row["target_char_start"])
        end = int(row["target_char_end"])

        enc = self.tokenizer(
            text,
            add_special_tokens=True,
            truncation=False,
            return_offsets_mapping=True
        )

        ids = enc["input_ids"]
        offsets = enc["offset_mapping"]

        targets = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < end and b > start
        ]

        if not targets:
            raise ValueError(
                f"Target alignment failed: {row['record_id']}"
            )

        capacity = self.max_length - 2
        first, last = targets[0], targets[-1]

        if last - first + 1 > capacity:
            raise ValueError("Target too long")

        left = max(1, first - capacity // 2)
        left = min(
            left,
            max(1, last - capacity + 1)
        )
        right = min(len(ids) - 1, left + capacity)

        if right - left < capacity:
            left = max(1, right - capacity)

        selected = list(range(left, right))

        input_ids = (
            [ids[0]]
            + [ids[i] for i in selected]
            + [ids[-1]]
        )

        target_mask = (
            [0.0]
            + [float(i in targets) for i in selected]
            + [0.0]
        )

        attention = [1] * len(input_ids)

        pad = self.max_length - len(input_ids)
        assert pad >= 0

        input_ids += [tokenizer.pad_token_id] * pad
        attention += [0] * pad
        target_mask += [0.0] * pad

        word = row["target_word"]

        allowed = [
            sense.startswith(word + ".")
            for sense in SENSES
        ]

        assert sum(allowed) == 2
        assert sum(target_mask) > 0

        return {
            "input_ids": torch.tensor(
                input_ids, dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                attention, dtype=torch.long
            ),
            "target_mask": torch.tensor(
                target_mask, dtype=torch.float32
            ),
            "class_mask": torch.tensor(
                allowed, dtype=torch.bool
            ),
            "labels": torch.tensor(
                sense_to_id[row["sense_id"]],
                dtype=torch.long
            )
        }


train_ds = LoRAWSDDataset(
    train_rows, tokenizer, max_length=64
)

val_ds = LoRAWSDDataset(
    val_rows, tokenizer, max_length=64
)

print("Train:", len(train_ds))
print("Validation:", len(val_ds))
print("Sample input shape:", train_ds[0]["input_ids"].shape)

Train: 506
Validation: 60
Sample input shape: torch.Size([64])


In [11]:
import gc
import torch

gc.collect()
torch.mps.empty_cache()

DEVICE = torch.device("mps")

model = model.to(DEVICE)
model.train()

sample = train_ds[0]

batch = {
    key: value.unsqueeze(0).to(DEVICE)
    for key, value in sample.items()
}

print("Device:", DEVICE)
print("Input shape:", batch["input_ids"].shape)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=2e-4
)

optimizer.zero_grad(set_to_none=True)

loss, logits = model(**batch)

assert torch.isfinite(loss), "Non-finite loss"

loss.backward()

torch.mps.synchronize()

print("Forward/backward successful.")
print("Loss:", float(loss.detach().cpu()))
print(
    "MPS allocated GB:",
    round(torch.mps.current_allocated_memory() / 1024**3, 3)
)
print(
    "MPS driver allocated GB:",
    round(torch.mps.driver_allocated_memory() / 1024**3, 3)
)

optimizer.zero_grad(set_to_none=True)
del batch, loss, logits
gc.collect()
torch.mps.empty_cache()

print("LoRA smoke test passed.")

Device: mps
Input shape: torch.Size([1, 64])
Forward/backward successful.
Loss: 0.6908578872680664
MPS allocated GB: 1.046
MPS driver allocated GB: 1.12
LoRA smoke test passed.


In [13]:
from torch.utils.data import DataLoader
import numpy as np
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    num_workers=0
)

optimizer = torch.optim.AdamW(
    [
        {
            "params": [
                p for p in model.encoder.parameters()
                if p.requires_grad
            ],
            "lr": 2e-4
        },
        {
            "params": list(model.classifier.parameters()),
            "lr": 5e-4
        }
    ],
    weight_decay=0.01
)

EPOCHS = 8
ACCUMULATION_STEPS = 8
PATIENCE = 3

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Gradient accumulation:", ACCUMULATION_STEPS)
print("Optimizer ready.")

Training batches: 506
Validation batches: 60
Gradient accumulation: 8
Optimizer ready.


In [15]:
from sklearn.metrics import accuracy_score, f1_score
import torch.nn.functional as F

SENSE_LABELS = list(range(len(SENSES)))

def move_batch(batch):
    return {
        key: value.to(DEVICE)
        for key, value in batch.items()
    }


def train_one_epoch(epoch):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    total_loss = 0.0
    correct = 0
    seen = 0

    for step, batch in enumerate(train_loader, start=1):
        batch = move_batch(batch)

        loss, logits = model(**batch)

        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss")

        (loss / ACCUMULATION_STEPS).backward()

        total_loss += float(loss.detach().cpu())

        predictions = logits.argmax(dim=1)
        correct += int(
            (predictions == batch["labels"]).sum().item()
        )
        seen += batch["labels"].size(0)

        if (
            step % ACCUMULATION_STEPS == 0
            or step == len(train_loader)
        ):
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                max_norm=1.0
            )

            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        if step % 100 == 0:
            print(
                f"Epoch {epoch} | "
                f"{step}/{len(train_loader)} batches"
            )

    return {
        "train_loss": total_loss / max(len(train_loader), 1),
        "train_accuracy": correct / max(seen, 1)
    }


@torch.inference_mode()
def validate():
    model.eval()

    predictions = []
    truth = []
    total_loss = 0.0

    for batch in val_loader:
        batch = move_batch(batch)

        loss, logits = model(**batch)

        total_loss += float(loss.detach().cpu())

        predictions.extend(
            logits.argmax(dim=1).cpu().tolist()
        )
        truth.extend(
            batch["labels"].cpu().tolist()
        )

    return {
        "val_loss": total_loss / max(len(val_loader), 1),
        "val_accuracy": accuracy_score(truth, predictions),
        "val_macro_f1": f1_score(
            truth,
            predictions,
            labels=SENSE_LABELS,
            average="macro",
            zero_division=0
        )
    }

print("Training and validation functions ready.")

Training and validation functions ready.


In [17]:
from pathlib import Path
import json
import copy
import time

OUT = ROOT / "artifacts" / "wsd_v2_lora"
OUT.mkdir(parents=True, exist_ok=True)

best_f1 = -1.0
best_epoch = None
wait = 0
history = []

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()

    train_metrics = train_one_epoch(epoch)
    val_metrics = validate()

    metrics = {
        "epoch": epoch,
        **train_metrics,
        **val_metrics,
        "duration_seconds": time.time() - start_time
    }

    history.append(metrics)

    print(
        f"\nEpoch {epoch:02d} | "
        f"Train Loss={metrics['train_loss']:.4f} | "
        f"Train Acc={metrics['train_accuracy']:.4f} | "
        f"Val Acc={metrics['val_accuracy']:.4f} | "
        f"Val Macro-F1={metrics['val_macro_f1']:.4f} | "
        f"Time={metrics['duration_seconds']:.1f}s"
    )

    if metrics["val_macro_f1"] > best_f1 + 1e-5:
        best_f1 = metrics["val_macro_f1"]
        best_epoch = epoch
        wait = 0

        # Save trainable LoRA adapters only.
        model.encoder.save_pretrained(
            OUT / "best_lora_adapter"
        )

        torch.save(
            model.classifier.state_dict(),
            OUT / "best_classifier_head.pt"
        )

        print("New best checkpoint saved.")

    else:
        wait += 1

    (OUT / "training_history.json").write_text(
        json.dumps(history, indent=2),
        encoding="utf-8"
    )

    if wait >= PATIENCE:
        print("Early stopping triggered.")
        break

print("\nTraining completed.")
print("Best epoch:", best_epoch)
print("Best validation Macro-F1:", best_f1)
print("Artifacts:", OUT)

Epoch 1 | 100/506 batches
Epoch 1 | 200/506 batches
Epoch 1 | 300/506 batches
Epoch 1 | 400/506 batches
Epoch 1 | 500/506 batches

Epoch 01 | Train Loss=0.5529 | Train Acc=0.7332 | Val Acc=0.9500 | Val Macro-F1=0.8741 | Time=13.7s
New best checkpoint saved.
Epoch 2 | 100/506 batches
Epoch 2 | 200/506 batches
Epoch 2 | 300/506 batches
Epoch 2 | 400/506 batches
Epoch 2 | 500/506 batches

Epoch 02 | Train Loss=0.1197 | Train Acc=0.9625 | Val Acc=1.0000 | Val Macro-F1=1.0000 | Time=12.9s
New best checkpoint saved.
Epoch 3 | 100/506 batches
Epoch 3 | 200/506 batches
Epoch 3 | 300/506 batches
Epoch 3 | 400/506 batches
Epoch 3 | 500/506 batches

Epoch 03 | Train Loss=0.0877 | Train Acc=0.9704 | Val Acc=1.0000 | Val Macro-F1=1.0000 | Time=13.0s
Epoch 4 | 100/506 batches
Epoch 4 | 200/506 batches
Epoch 4 | 300/506 batches
Epoch 4 | 400/506 batches
Epoch 4 | 500/506 batches

Epoch 04 | Train Loss=0.0816 | Train Acc=0.9783 | Val Acc=1.0000 | Val Macro-F1=1.0000 | Time=13.0s
Epoch 5 | 100/506 batc

In [19]:
import torch
from peft import PeftModel
from transformers import AutoModel

# Release the epoch-5 model before reloading.
model.to("cpu")
del model
del encoder
import gc
gc.collect()

if torch.backends.mps.is_available():
    torch.mps.empty_cache()

base = AutoModel.from_pretrained(
    MODEL_NAME,
    attn_implementation="eager"
)

best_encoder = PeftModel.from_pretrained(
    base,
    OUT / "best_lora_adapter",
    is_trainable=False
)

best_model = LoRAWSDClassifier(
    encoder=best_encoder,
    num_classes=len(SENSES)
)

head_state = torch.load(
    OUT / "best_classifier_head.pt",
    map_location="cpu",
    weights_only=True
)

best_model.classifier.load_state_dict(head_state)
best_model.to(DEVICE)
best_model.eval()

print("Best LoRA checkpoint reloaded.")
print("Selected epoch:", best_epoch)
print("Device:", DEVICE)

Best LoRA checkpoint reloaded.
Selected epoch: 2
Device: mps


In [21]:
import json
from pathlib import Path

CHALLENGE_PATH = (
    ROOT / "artifacts" / "wsd_robustness"
    / "challenge_records.jsonl"
)

with CHALLENGE_PATH.open(encoding="utf-8") as f:
    challenge_rows = [
        json.loads(line)
        for line in f if line.strip()
    ]

assert len(challenge_rows) == 36

challenge_ds = LoRAWSDDataset(
    challenge_rows,
    tokenizer,
    max_length=64
)

print("Challenge examples:", len(challenge_ds))
print("Classes:", len(SENSES))
print("Model device:", DEVICE)

Challenge examples: 36
Classes: 12
Model device: mps


In [23]:
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

challenge_loader = DataLoader(
    challenge_ds,
    batch_size=1,
    shuffle=False,
    num_workers=0
)

best_model.eval()

predictions = []
truth = []

with torch.inference_mode():
    for batch in challenge_loader:
        batch = {
            k: v.to(DEVICE)
            for k, v in batch.items()
        }

        logits = best_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            target_mask=batch["target_mask"],
            class_mask=batch["class_mask"]
        )

        pred = logits.argmax(dim=1)

        predictions.extend(pred.cpu().tolist())
        truth.extend(batch["labels"].cpu().tolist())

y_true_lora = [SENSES[i] for i in truth]
y_pred_lora = [SENSES[i] for i in predictions]

print("=== LoRA CHALLENGE EVALUATION ===")

print(
    "Accuracy:",
    accuracy_score(y_true_lora, y_pred_lora)
)

print(
    "Macro Precision:",
    precision_score(
        y_true_lora, y_pred_lora,
        labels=SENSES,
        average="macro",
        zero_division=0
    )
)

print(
    "Macro Recall:",
    recall_score(
        y_true_lora, y_pred_lora,
        labels=SENSES,
        average="macro",
        zero_division=0
    )
)

print(
    "Macro-F1:",
    f1_score(
        y_true_lora, y_pred_lora,
        labels=SENSES,
        average="macro",
        zero_division=0
    )
)

print("\nClassification report:")
print(classification_report(
    y_true_lora,
    y_pred_lora,
    labels=SENSES,
    zero_division=0
))

=== LoRA CHALLENGE EVALUATION ===
Accuracy: 1.0
Macro Precision: 1.0
Macro Recall: 1.0
Macro-F1: 1.0

Classification report:
              precision    recall  f1-score   support

      bank.1       1.00      1.00      1.00         3
      bank.2       1.00      1.00      1.00         3
  interest.1       1.00      1.00      1.00         3
  interest.2       1.00      1.00      1.00         3
     plant.1       1.00      1.00      1.00         3
     plant.2       1.00      1.00      1.00         3
     point.1       1.00      1.00      1.00         3
     point.2       1.00      1.00      1.00         3
     right.1       1.00      1.00      1.00         3
     right.2       1.00      1.00      1.00         3
     table.1       1.00      1.00      1.00         3
     table.2       1.00      1.00      1.00         3

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36



In [25]:
print("=== LoRA ERROR ANALYSIS ===")

errors = []

for row, gold, pred in zip(
    challenge_rows,
    y_true_lora,
    y_pred_lora
):
    if gold != pred:
        errors.append({
            "sentence": row["text_en"],
            "gold": gold,
            "predicted": pred
        })

for error in errors:
    print("\nSentence:", error["sentence"])
    print("Gold:", error["gold"])
    print("Predicted:", error["predicted"])

print("\nTotal errors:", len(errors))
print("Correct:", len(challenge_rows) - len(errors))
print("Total:", len(challenge_rows))

results_path = OUT / "lora_challenge_results.json"

results_path.write_text(
    json.dumps({
        "checkpoint_epoch": best_epoch,
        "evaluation_set": "assistant_reviewed_challenge_v1",
        "total": len(challenge_rows),
        "correct": len(challenge_rows) - len(errors),
        "errors": errors
    }, indent=2),
    encoding="utf-8"
)

print("\nResults saved:", results_path)

=== LoRA ERROR ANALYSIS ===

Total errors: 0
Correct: 36
Total: 36

Results saved: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/artifacts/wsd_v2_lora/lora_challenge_results.json
